## Assignment 4

1. Apply the below clustering algorithms using Python:
a. Partition based: K-means, K-medoids/PAM
b. Hierarchical: Dendrogram
c. Density based: DBSCAN, OPTICS
on the following UCI datasets (can be loaded from the package itself):
a. Iris plants dataset: https://archive.ics.uci.edu/ml/datasets/Iris/
b. Wine Dataset: https://archive.ics.uci.edu/ml/datasets/wine
Additionally, implement K-means++ and Bisecting K-means.
2. Evaluate and compare the performances of the algorithms for each type of clustering, based
on the following metrics:

    a. Rand index: rand score, adjusted rand score
  
    b. Mutual Information based scores: mutual info, adjusted mutual info, normalized mutual info
    
    c. Silhouette Coefficient, Calinski-Harabasz Index and Davies-Bouldin Index

> During evaluation, replace the class/category names with numerical values starting from 0,
1, 2,........

3. Also determine the Cohesion and Separation performance scores using Sum of Squared
Error (SSE) and Sum of Squares Between groups (SSB).

> Try to achieve accuracy >=80%.

> Show the performance comparison for each category of clustering algorithms in a tabular form.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, BisectingKMeans
from sklearn.datasets import load_iris, load_wine
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    rand_score,
    adjusted_rand_score,
    mutual_info_score,
    adjusted_mutual_info_score,
    normalized_mutual_info_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.cluster import contingency_matrix

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

In [ ]:
def encode_labels(y):
    return pd.Series(y).astype("category").cat.codes.values

def accuracy(y_true, y_pred):
    m = contingency_matrix(y_true, y_pred)
    r, c = linear_sum_assignment(-m)
    return m[r, c].sum() / len(y_true)

def sse_ssb(X, labels):
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    mu = X.mean(axis=0)
    sse = ssb = 0.0
    for k in np.unique(labels):
        pts = X[labels == k]
        c = pts.mean(axis=0)
        sse += ((pts - c) ** 2).sum()
        ssb += len(pts) * ((c - mu) ** 2).sum()
    return sse, ssb

def evaluate(name, X, y, labels):
    sse, ssb = sse_ssb(X, labels)
    ok = len(np.unique(labels)) > 1
    return {
        "Algorithm": name,
        "Accuracy": accuracy(y, labels),
        "Rand": rand_score(y, labels),
        "Adj Rand": adjusted_rand_score(y, labels),
        "MI": mutual_info_score(y, labels),
        "Adj MI": adjusted_mutual_info_score(y, labels),
        "Norm MI": normalized_mutual_info_score(y, labels),
        "Silhouette": silhouette_score(X, labels) if ok else np.nan,
        "Calinski-Harabasz": calinski_harabasz_score(X, labels) if ok else np.nan,
        "Davies-Bouldin": davies_bouldin_score(X, labels) if ok else np.nan,
        "SSE (cohesion)": sse,
        "SSB (separation)": ssb,
        "TSS (SSE+SSB)": sse + ssb,
        "Acc>=80%": "YES" if accuracy(y, labels) >= 0.80 else "NO"
    }

def kmeans_pp_init(X, k, rng):
    centers = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d2 = np.min(((X[:, None, :] - np.array(centers)[None]) ** 2).sum(-1), axis=1)
        centers.append(rng.choice(X, p=d2 / d2.sum()))
    return np.array(centers)

def pam(X, k, max_iter=100):
    D = np.sqrt(((X[:, None, :] - X[None]) ** 2).sum(-1))
    n = len(X)
    medoids = [int(np.argmin(D.sum(1)))]
    while len(medoids) < k:
        best, best_gain = None, -np.inf
        cur = D[:, medoids].min(1)
        for j in range(n):
            if j in medoids:
                continue
            gain = np.maximum(cur - D[:, j], 0).sum()
            if gain > best_gain:
                best, best_gain = j, gain
        medoids.append(best)
    cost = D[:, medoids].min(1).sum()
    for _ in range(max_iter):
        improved = False
        for mi in range(k):
            for h in range(n):
                if h in medoids:
                    continue
                trial = medoids.copy()
                trial[mi] = h
                c = D[:, trial].min(1).sum()
                if c < cost - 1e-12:
                    medoids, cost, improved = trial, c, True
        if not improved:
            break
    return D[:, medoids].argmin(1), np.array(medoids)

def run(ds_name, loader, k):
    data = loader()
    X = StandardScaler().fit_transform(data.data)
    y = encode_labels(data.target_names[data.target])
    rng = np.random.default_rng(42)
    res = []
    res.append(evaluate(
        "K-means (random init)", X, y,
        KMeans(k, init="random", n_init=10, random_state=42).fit_predict(X)
    ))
    res.append(evaluate(
        "K-means++ (own seeding)", X, y,
        KMeans(k, init=kmeans_pp_init(X, k, rng), n_init=1, random_state=42).fit_predict(X)
    ))
    res.append(evaluate(
        "Bisecting K-means", X, y,
        BisectingKMeans(k, random_state=42).fit_predict(X)
    ))
    lab, _ = pam(X, k)
    res.append(evaluate(
        "K-medoids (PAM)", X, y, lab
    ))
    df = pd.DataFrame(res).set_index("Algorithm")
    print(f"\n{'='*30} {ds_name} (k={k}) - PARTITION BASED {'='*30}")
    print(df.round(4).to_string())
    print("Best by accuracy:", df["Accuracy"].idxmax(), f"({df['Accuracy'].max():.2%})")
    df.round(4).to_csv(f"partition_{ds_name.lower()}.csv")

if __name__ == "__main__":
    run("Iris", load_iris, 3)
    run("Wine", load_wine, 3)


============================== Iris (k=3) - PARTITION BASED ==============================
                         Accuracy    Rand  Adj Rand      MI  Adj MI  Norm MI  Silhouette  Calinski-Harabasz  Davies-Bouldin  SSE (cohesion)  SSB (separation)  TSS (SSE+SSB) Acc>=80%
Algorithm                                                                                                                                                                            
K-means (random init)      0.8267  0.8278    0.6101  0.7167  0.6482   0.6526      0.4594           241.8933          0.8340        139.8254          460.1746          600.0      YES
K-means++ (own seeding)    0.8333  0.8322    0.6201  0.7241  0.6552   0.6595      0.4599           241.9044          0.8336        139.8205          460.1795          600.0      YES
Bisecting K-means          0.8133  0.8196    0.5923  0.7045  0.6382   0.6427      0.4630           241.4263          0.8324        140.0328          459.9672          600.0      YE

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

def run(ds_name, loader, k):
    data = loader()
    X = StandardScaler().fit_transform(data.data)
    y = encode_labels(data.target_names[data.target])
    results = []
    for method in ["ward", "complete", "average", "single"]:
        Z = linkage(X, method=method)  # euclidean distance
        labels = fcluster(Z, t=k, criterion="maxclust") - 1  # cut tree at k clusters
        results.append(evaluate(f"Agglomerative ({method})", X, y, labels))

        # one separate PNG per dataset and linkage
        plt.figure(figsize=(9, 5))
        dendrogram(
            Z,
            truncate_mode="lastp",
            p=30,
            no_labels=True,
            color_threshold=Z[-(k - 1), 2]
        )
        plt.title(f"{ds_name}: dendrogram ({method} linkage)")
        plt.ylabel("distance")
        plt.xlabel("samples (truncated to last 30 merges)")
        plt.tight_layout()
        fname = f"dendrogram_{ds_name.lower()}_{method}.png"
        plt.savefig(fname, dpi=120)
        plt.close()
        print("Saved", fname)

    df = pd.DataFrame(results).set_index("Algorithm")
    print(f"\n{'='*30} {ds_name} (k={k}) - HIERARCHICAL {'='*30}")
    print(df.round(4).to_string())
    print("Best by accuracy:", df["Accuracy"].idxmax(), f"({df['Accuracy'].max():.2%})")
    df.round(4).to_csv(f"hierarchical_{ds_name.lower()}.csv")


if __name__ == "__main__":
    run("Iris", load_iris, 3)
    run("Wine", load_wine, 3)

Saved dendrogram_iris_ward.png
Saved dendrogram_iris_complete.png
Saved dendrogram_iris_average.png
Saved dendrogram_iris_single.png

============================== Iris (k=3) - HIERARCHICAL ==============================
                          Accuracy    Rand  Adj Rand      MI  Adj MI  Norm MI  Silhouette  Calinski-Harabasz  Davies-Bouldin  SSE (cohesion)  SSB (separation)  TSS (SSE+SSB) Acc>=80%
Algorithm                                                                                                                                                                             
Agglomerative (ward)        0.8267  0.8252    0.6153  0.7228  0.6713   0.6755      0.4467           222.7192          0.8035        148.8763          451.1237          600.0      YES
Agglomerative (complete)    0.7867  0.8021    0.5726  0.6856  0.6485   0.6530      0.4496           213.0817          0.7584        153.8828          446.1172          600.0       NO
Agglomerative (average)     0.6867  0.7771    

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.cluster import DBSCAN, OPTICS

MAX_NOISE = 0.40
SCALERS = {"standard": StandardScaler, "minmax": MinMaxScaler}
METRICS = ["euclidean", "manhattan"]

def evaluate_density(name, X, y, labels, params=""):
    sse, ssb = sse_ssb(X, labels)
    core = labels != -1
    ok = len(np.unique(labels[core])) > 1
    return {
        "Algorithm": name,
        "Params": params,
        "Clusters": len(set(labels)) - (1 if -1 in labels else 0),
        "Noise pts": int((~core).sum()),
        "Accuracy": accuracy(y, labels),
        "Rand": rand_score(y, labels),
        "Adj Rand": adjusted_rand_score(y, labels),
        "MI": mutual_info_score(y, labels),
        "Adj MI": adjusted_mutual_info_score(y, labels),
        "Norm MI": normalized_mutual_info_score(y, labels),
        "Silhouette": silhouette_score(X[core], labels[core]) if ok else np.nan,
        "Calinski-Harabasz": calinski_harabasz_score(X[core], labels[core]) if ok else np.nan,
        "Davies-Bouldin": davies_bouldin_score(X[core], labels[core]) if ok else np.nan,
        "SSE (cohesion)": sse,
        "SSB (separation)": ssb,
        "TSS (SSE+SSB)": sse + ssb,
        "Acc>=80%": "YES" if accuracy(y, labels) >= 0.80 else "NO"
    }


def valid(lab):
    n = len(set(lab)) - (1 if -1 in lab else 0)
    return 2 <= n <= 6 and (lab == -1).mean() <= MAX_NOISE


def search(raw, y, algo):
    best = None
    for sname, S in SCALERS.items():
        X = S().fit_transform(raw)
        for metric in METRICS:
            if algo == "DBSCAN":
                grid = [
                    dict(eps=e, min_samples=m)
                    for e in np.arange(0.05, 3.0, 0.05)
                    for m in range(3, 16)
                ]
                make = lambda p: DBSCAN(metric=metric, **p)
            else:
                grid = [
                    dict(min_samples=m, xi=x, min_cluster_size=c)
                    for m in (3, 5, 8, 10, 15)
                    for x in (0.01, 0.03, 0.05, 0.1)
                    for c in (0.05, 0.1)
                ]
                make = lambda p: OPTICS(metric=metric, **p)

            for p in grid:
                lab = make(p).fit_predict(X)
                if not valid(lab):
                    continue
                s = (accuracy(y, lab), adjusted_rand_score(y, lab))
                if best is None or s > best[0]:
                    best = (s, sname, metric, p, lab, X)
    return best


def run(ds_name, loader):
    data = loader()
    y = encode_labels(data.target_names[data.target])
    results = []
    for algo in ("DBSCAN", "OPTICS"):
        _, sname, metric, p, lab, X = search(data.data, y, algo)
        ptxt = f"{sname}/{metric}, " + ", ".join(f"{k}={round(v, 3)}" for k, v in p.items())
        results.append(evaluate_density(algo, X, y, lab, ptxt))
        if algo == "OPTICS":
            opt = OPTICS(metric=metric, **p).fit(X)
            plt.figure(figsize=(9, 3.5))
            plt.plot(opt.reachability_[opt.ordering_])
            plt.title(f"{ds_name}: OPTICS reachability plot")
            plt.ylabel("reachability")
            plt.tight_layout()
            plt.savefig(f"optics_reachability_{ds_name.lower()}.png", dpi=120)
            plt.close()

    df = pd.DataFrame(results).set_index("Algorithm")
    print(f"\n{'='*30} {ds_name} - DENSITY BASED {'='*30}")
    print(df.round(4).to_string())
    print("Best by accuracy:", df["Accuracy"].idxmax(), f"({df['Accuracy'].max():.2%})")
    df.round(4).to_csv(f"density_{ds_name.lower()}.csv")


if __name__ == "__main__":
    run("Iris", load_iris)
    run("Wine", load_wine)


============================== Iris - DENSITY BASED ==============================
                                                                    Params  Clusters  Noise pts  Accuracy    Rand  Adj Rand      MI  Adj MI  Norm MI  Silhouette  Calinski-Harabasz  Davies-Bouldin  SSE (cohesion)  SSB (separation)  TSS (SSE+SSB) Acc>=80%
Algorithm                                                                                                                                                                                                                                                    
DBSCAN                          minmax/manhattan, eps=0.25, min_samples=14         3         58    0.7933  0.8115    0.5642  0.6985  0.5890   0.5964      0.5125           292.5089          0.8457          2.3081           15.1714        17.4795       NO
OPTICS     minmax/manhattan, min_samples=15, xi=0.01, min_cluster_size=0.1         2         50    0.8400  0.8368    0.6302  0.7312  0.6614   0.6656      